# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using output/cohort.csv produced by 01_build_cohort.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8b_vascular_signal"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8b_vascular_signal_H5.csv (38341, 79)
label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,VASC_MED_COUNT,GAIT_ANY,CDR_ADL_COG_RATIO,CDR_ADL_MINUS_COG,STROKE_x_EXEC,STROKDEC,STROKMUL,NACCMOM,NACCDAD,SIVDFIND
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,0.0,0,0.0,-0.5,-0.0,NaN,NaN,NaN,NaN,NaN
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,2.0,0,0.0,-0.5,-0.0,0.0,0.0,1.0,1.0,0.0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,1.0,1,0.0,-0.5,-0.0,NaN,NaN,1.0,0.0,NaN
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,0.0,0,NaN,0.0,-0.0,0.0,0.0,0.0,0.0,0.0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,0.0,0,NaN,0.0,-0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [4]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        #X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [5]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline", "NACCUDSD"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # What is dropped is marked only after computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14537
label
NoDementia_Hy    11083
AD                3013
PD_LBD             289
Vascular           152
Name: count, dtype: int64
Train: (11629, 72) Test: (2908, 72)
Train label counts:
 label
NoDementia_Hy    8866
AD               2410
PD_LBD            231
Vascular          122
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                603
PD_LBD             58
Vascular           30
Name: count, dtype: int64


In [6]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [7]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )

Preprocessing pipelines (sparse for LR/boosting)

In [8]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print("Num cols:", num_cols, "Cat cols:", cat_cols)

# print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
# print(sorted([c for c in X_train.columns]))



Num cols: 69 Cat cols: 3
Num cols: ['NACCAGE', 'EDUC', 'NACCMMSE', 'NACCMOCA', 'TRAILA', 'TRAILB', 'ANIMALS', 'BOSTON', 'CDRSUM', 'CDRGLOB', 'NACCMMSE_missing', 'NACCMOCA_missing', 'TRAILA_missing', 'TRAILB_missing', 'CDRSUM_missing', 'COGFLUC', 'BEVHALL', 'BEREM', 'PARKSIGN', 'BRADY', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SLOWINGL', 'SLOWINGR', 'PARKGAIT', 'GAITDIS', 'DIABETES', 'HYPERTEN', 'HYPERCHO', 'CVAFIB', 'CBSTROKE', 'CBTIA', 'CVCHF', 'CVANGINA', 'CVBYPASS', 'CVANGIO', 'CVHVALVE', 'CVOTHR', 'APNEA', 'DEP2YRS', 'DIABET', 'POSTINST', 'NACCGDS', 'DIGBACCT', 'DIGBACLS', 'CVD', 'VASC', 'CVDCOG', 'CVDIMAG', 'IMAGMWMH', 'IMAGEWMH', 'FOCLSYM', 'FOCLSIGN', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'STROKE_TIA_ANY', 'VASC_MED_COUNT', 'GAIT_ANY', 'CDR_ADL_COG_RATIO', 'CDR_ADL_MINUS_COG', 'STROKE_x_EXEC', 'STROKDEC', 'STROKMUL', 'NACCMOM', 'NACCDAD', 'SIVDFIND'] Cat cols: ['SEX', 'RACE', 'HISPANIC']


Model A: Logistic Regression (final settings)

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [9]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6575    0.7164    0.6857       603
NoDementia_Hy     0.9306    0.8958    0.9129      2217
       PD_LBD     0.4762    0.6897    0.5634        58
     Vascular     0.3333    0.3667    0.3492        30

     accuracy                         0.8490      2908
    macro avg     0.5994    0.6671    0.6278      2908
 weighted avg     0.8588    0.8490    0.8530      2908

Balanced accuracy: 0.6671362229030295
[[ 432  137   23   11]
 [ 201 1986   20   10]
 [  13    4   40    1]
 [  11    7    1   11]]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# 0) Labels / weights
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# Slightly reduced vascular weight to help precision/F1
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# 1) Train / validation split from train only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Candidate model settings
#   Elastic Net logistic regression
model_settings = [
    {
        "name": "LR_L2_C0.1",
        # "penalty": "l2",
        # "solver": "lbfgs",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0,
    },
    {
        "name": "LR_L2_C0.05",
        # "penalty": "l2",
        # "solver": "lbfgs",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# applying vascular threshold override
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# 3) Validation search:
#    chooseing model + threshold by vascular F1
#    macro F1, balanced accuracy
search_rows = []

for cfg in model_settings:
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            # max_iter=12000,
            max_iter=50000,
            tol=1e-5,          # tighter convergence tolerance
            class_weight=custom_w,
            C=cfg["C"],
            # penalty=cfg["penalty"],
            solver=cfg["solver"],
            l1_ratio=cfg["l1_ratio"],
            # n_jobs=None
        ))
    ])

    clf.fit(X_tr, y_tr)

    proba_val = clf.predict_proba(X_val)
    class_order = clf.named_steps["model"].classes_

    for T in thresholds:
        preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

        # overall metrics
        ba = balanced_accuracy_score(y_val, preds_val)
        macro_f1 = f1_score(y_val, preds_val, average="macro")

        # per-class metrics
        p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
            y_val, preds_val, labels=labels, zero_division=0
        )

        metric_map = {
            lab: {"precision": p, "recall": r, "f1": f, "support": s}
            for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
        }

        vprec = metric_map["Vascular"]["precision"]
        vrec  = metric_map["Vascular"]["recall"]
        vf1   = metric_map["Vascular"]["f1"]

        search_rows.append({
            "model_name": cfg["name"],
            # "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "val_bal_acc": ba,
            "val_macro_f1": macro_f1,
            "val_vasc_precision": vprec,
            "val_vasc_recall": vrec,
            "val_vasc_f1": vf1,
        })

search_df = pd.DataFrame(search_rows)

# Sorting by vascular F1 first, then macro F1, then BA
search_df = search_df.sort_values(
    ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Top validation settings:")
display(search_df.head(10))

best_row = search_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"val VascF1={best_row['val_vasc_f1']:.4f} | "
    f"val MacroF1={best_row['val_macro_f1']:.4f} | "
    f"val BA={best_row['val_bal_acc']:.4f} | "
    f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
    f"val VascRec={best_row['val_vasc_recall']:.4f}"
)



c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:

# 4) Refiting best model on full train
best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        tol=1e-5,          # tighter convergence tolerance
        class_weight=custom_w,
        C=best_cfg["C"],
        # penalty=best_cfg["penalty"],
        solver=best_cfg["solver"],
        l1_ratio=best_cfg["l1_ratio"],
        # n_jobs=None
    ))
])


clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# 5) Final test evaluation
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6587    0.7297    0.6924       603
NoDementia_Hy     0.9319    0.8954    0.9133      2217
       PD_LBD     0.5139    0.6379    0.5692        58
     Vascular     0.2632    0.3333    0.2941        30

     accuracy                         0.8501      2908
    macro avg     0.5919    0.6491    0.6172      2908
 weighted avg     0.8600    0.8501    0.8542      2908

Balanced accuracy: 0.6490758396745991
Macro F1: 0.6172475380658718
Weighted F1: 0.8542174502330312
[[ 440  132   16   15]
 [ 201 1985   19   12]
 [  14    6   37    1]
 [  13    7    0   10]]

Test vascular metrics:
{'precision': np.float64(0.2631578947368421), 'recall': np.float64(0.3333333333333333), 'f1': np.float64(0.29411764705882354), 'support': np.int64(30)}


In [11]:

# 4) Refiting best model on full train

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

# applying vascular threshold override
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        tol=1e-5,          # tighter convergence tolerance
        class_weight=custom_w,
        C=0.05,
        # penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        # n_jobs=None
    ))
])

best_T = 0.360

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# 5) Final test evaluation
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])


=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6575    0.7164    0.6857       603
NoDementia_Hy     0.9336    0.8940    0.9134      2217
       PD_LBD     0.4699    0.6724    0.5532        58
     Vascular     0.2444    0.3667    0.2933        30

     accuracy                         0.8473      2908
    macro avg     0.5764    0.6624    0.6114      2908
 weighted avg     0.8600    0.8473    0.8526      2908

Balanced accuracy: 0.6623748180844645
Macro F1: 0.6114007909272151
Weighted F1: 0.8525787246450384
[[ 432  130   23   18]
 [ 200 1982   20   15]
 [  14    4   39    1]
 [  11    7    1   11]]

Test vascular metrics:
{'precision': np.float64(0.24444444444444444), 'recall': np.float64(0.36666666666666664), 'f1': np.float64(0.29333333333333333), 'support': np.int64(30)}


c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [12]:
import joblib
import pandas as pd

joblib.dump(
    {
        "model": clf_best,
        "threshold": best_T,
        "class_order": list(class_order),
        "feature_cols": list(X_train.columns)
    },
    "../output/dementia_classifier_bundle.joblib"
)

print("Model bundle saved.")

# Predicting on test set
proba = clf_best.predict_proba(X_test)
preds = predict_with_vascular_threshold(proba, class_order, best_T)

# Exporting predictions with true NACCID from df4
clf_preds = pd.DataFrame({
    "NACCID": df4.loc[X_test.index, "NACCID"].values,
    "predicted_class": preds
})

for i, cls in enumerate(class_order):
    clf_preds[f"prob_{cls}"] = proba[:, i]

clf_preds["vascular_threshold"] = best_T

clf_preds.to_csv("../output/classifier_predictions.csv", index=False)

print("Saved ../output/classifier_predictions.csv")
display(clf_preds.head())

Model bundle saved.
Saved ../output/classifier_predictions.csv


,NACCID,predicted_class,prob_AD,prob_NoDementia_Hy,prob_PD_LBD,prob_Vascular,vascular_threshold
0,NACC056862,NoDementia_Hy,0.084440,0.908303,0.002438,0.004819,0.36
1,NACC774813,NoDementia_Hy,0.073318,0.921054,0.003956,0.001673,0.36
2,NACC989206,NoDementia_Hy,0.063252,0.932215,0.001510,0.003023,0.36
3,NACC735141,AD,0.783594,0.090478,0.085194,0.040735,0.36
4,NACC252819,AD,0.572741,0.374559,0.012795,0.039905,0.36
